# MAPNAI: GPU runs on Kaggle (LLM-AggreFact)

Same runs and protocol as `mapnai_gpu_runs.ipynb` (Colab), adapted for Kaggle:

| Part | What | Needed for | Time on a T4 (estimate) |
|---|---|---|---|
| 1 | MiniCheck-Flan-T5-Large on the **test** split (29,320 pairs) | **S1** (reproduce the published 75.0) | 3.2 h (measured, 2026-10-31) |
| 2 | MiniCheck-Flan-T5-Large on the **dev** split (30,420 pairs) | CAFG tuning (S3) | 3.5 to 4 h |
| 2b | MiniCheck-DeBERTa-v3-Large on test + dev (optional, off by default) | cheaper baseline | about half of Part 1 |

**How to run it:** use **Save Version → Save & Run All (Commit)**. It runs in the background (you can close the browser) for up to 12 hours.
The notebook stops scoring by itself before the 12-hour limit (`TIME_BUDGET_H`), and always writes the results zip at the end.

**Settings (right-hand panel):** Accelerator **GPU T4 x2** (not P100), Internet **on**, and the secret **HF_TOKEN** attached
(Add-ons → Secrets). The full checklist is in `research/colab/INSTRUCTIONS.md`.

**If a run stopped before everything was scored:** open the notebook, **Add Input → Your Work → this notebook's output**, and
Save & Run All again. Cell 0.2 copies the finished rows in, and only the missing rows are scored.

Everything is pinned (code commit, model revisions, dataset revision) and checked by sha256.
The protocol is the one in `research/PREREG.md` §2 (threshold 0.5, no tuning, MiniCheck's own chunking).

**GPU memory:** a few documents have long chunks (up to 2,048 tokens), and a batch of 16 of them ran a T4 out of memory
on the first run (dev split, row ~4,700). If that happens now, the shard is retried at batch size 4, then 1.
Batch size only changes padding, which is masked, so the scores are the same up to float rounding. Rows scored at a
smaller batch carry a `bs` field in the cache file.

## Part 0: Setup

In [ ]:
# 0.1  Which GPU did Kaggle give you?  (Settings > Accelerator > GPU T4 x2.)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 0.2  Folders, options, and resume from an earlier run's output (if you attached one as input).
import os, glob, json, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must be set before torch uses the GPU
START = time.time()
TIME_BUDGET_H = 11.0            # stop scoring after this many hours, so the 12 h Kaggle limit never kills the run
RUN_DEBERTA = False             # Part 2b (optional); set True to also score MiniCheck-DeBERTa
TAG = 'kaggle-2026-10-31'       # do not change between runs: it is how rows are resumed
WORK = '/kaggle/working'        # saved as the notebook's output
DATA = '/kaggle/temp/data'      # not saved (keeps the output small)
CKPTS = '/kaggle/temp/ckpts'
for d in (f'{WORK}/cache', f'{WORK}/outputs', DATA, CKPTS):
    os.makedirs(d, exist_ok=True)

# Merge finished rows from any attached earlier output (same TAG), keeping one line per row.
for prev in glob.glob(f'/kaggle/input/**/e20_*_{TAG}.jsonl', recursive=True):
    dst = f'{WORK}/cache/{os.path.basename(prev)}'
    rows = {}
    for f in (prev, dst):
        if os.path.exists(f):
            for l in open(f, encoding='utf-8'):
                if l.strip():
                    r = json.loads(l)
                    rows[r['row']] = l if l.endswith('\n') else l + '\n'
    with open(dst, 'w', encoding='utf-8') as fh:
        fh.writelines(rows[k] for k in sorted(rows))
    print(f'resumed {len(rows)} rows from {prev}')
print('working folder:', WORK)

In [ ]:
# 0.3  Pinned packages (about 2 minutes). Needs Internet on.
!pip install -q "transformers==4.57.6" "tokenizers==0.22.2" "accelerate==1.10.1" "datasets==3.6.0" "nltk==3.9.1" "huggingface_hub==0.36.2" "scikit-learn==1.7.2"
!pip install -q --no-deps "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@b58b9fa69acbd1015ec970fa65dd752413a053d2"
import nltk
nltk.download('punkt_tab', quiet=True)
os.makedirs(os.path.expanduser('~/nltk_data/tokenizers/punkt'), exist_ok=True)   # stops minicheck from fetching the old pickle "punkt"
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU', torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability(0) >= (7, 0), 'P100 is not supported by this torch build: choose GPU T4 x2'

In [ ]:
# 0.4  Log in to Hugging Face with the Kaggle secret HF_TOKEN (Add-ons > Secrets; tick "attached" for this notebook).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
# 0.5  Download the pinned dataset and checkpoints, and verify every file by sha256.
import hashlib
from huggingface_hub import snapshot_download, HfApi

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()

DATA_REV = '981dfd0bd8e58e7238a9ab92b2e6ea44bce918e4'
data_dir = snapshot_download('lytang/LLM-AggreFact', repo_type='dataset', revision=DATA_REV,
                             allow_patterns=['data/*.parquet'], local_dir=DATA)
EXPECTED = {'data/test-00000-of-00001.parquet': 'ba6976c3da990a9d7b11670efaf4e3c93c1bd0530061ec742afd6b2d5bce698c',
            'data/dev-00000-of-00001.parquet': 'ca9a5bda9561bbc6f7327ef81ab41727fa10e96f96e9dc74ef32ba3af3c1d8a8'}
for fn, want in EXPECTED.items():
    assert sha256(f'{data_dir}/{fn}') == want, f'{fn} does not match the pinned dataset'
print('dataset OK')

# minicheck loads the checkpoints by name at revision "main"; we check that "main" is still the pinned commit.
MODELS = {'lytang/MiniCheck-Flan-T5-Large': ('96eafd01cee2d16cf81aaa2fb226b14f422a37b3',
                                              '41291881e13c6235ed47149cec903bee9493e45d9d7325587a9fa2e266c526c0'),
          'lytang/MiniCheck-DeBERTa-v3-Large': ('2f2d01a54fa022a7ffadb76260e1ea8bc88c82bb',
                                                 '32ced784f4831d5f324a072e43b96d4072cac4bf41042bc42243605ffbbccbdf')}
for repo, (rev, bin_sha) in MODELS.items():
    live = HfApi().model_info(repo).sha
    assert live == rev, f'{repo}: main moved to {live}; stop and tell Claude'
    path = snapshot_download(repo, revision=rev, cache_dir=CKPTS)
    assert sha256(f'{path}/pytorch_model.bin') == bin_sha, f'{repo}: weights differ from the pinned file'
print('checkpoints OK')

In [ ]:
# 0.6  The scoring and summary code (same protocol as research/experiments/e20_aggrefact_eval.py).
import json, time
import numpy as np
import pandas as pd

def load_split(split):
    return pd.read_parquet(f'{DATA}/data/{split}-00000-of-00001.parquet').reset_index(drop=True)

def cache_path(checker, split):
    return f'{WORK}/cache/e20_{checker}_{split}_{TAG}.jsonl'

def score_shard(scorer, df, ids, batch_size):
    """Scores one shard; on a GPU out-of-memory error, retries it at batch size 4, then 1.
    Batch size only changes (masked) padding, so the scores match up to float rounding."""
    import gc, torch
    for bs in (batch_size, 4, 1):
        if bs > batch_size:
            continue
        scorer.model.batch_size = bs
        try:
            _, probs, _, _ = scorer.score(docs=[df.doc[i] for i in ids], claims=[df.claim[i] for i in ids])
            return probs, bs
        except torch.OutOfMemoryError:
            print(f'  out of GPU memory at batch size {bs} (rows {ids[0]}..{ids[-1]}); retrying smaller', flush=True)
        finally:
            scorer.model.batch_size = batch_size
        gc.collect()                 # outside the except block, so the failed batch's tensors are really released
        torch.cuda.empty_cache()
    raise RuntimeError('out of GPU memory even at batch size 1')

def score_split(checker, split, shard=256, batch_size=16):
    """Scores every row; rows already in the cache file are skipped (safe to re-run after a disconnect)."""
    from minicheck.minicheck import MiniCheck
    name = {'minicheck-ft5': 'flan-t5-large', 'minicheck-deberta': 'deberta-v3-large'}[checker]
    df, path = load_split(split), cache_path(checker, split)
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)['row'] for l in open(path, encoding='utf-8')}
    todo = [i for i in range(len(df)) if i not in done]
    print(f'{checker} {split}: {len(done)} done, {len(todo)} to score')
    if not todo:
        return
    if time.time() - START > TIME_BUDGET_H * 3600:
        print('time budget already used: skipping')
        return
    scorer = MiniCheck(model_name=name, cache_dir=CKPTS, batch_size=batch_size)
    t0 = time.time()
    with open(path, 'a', encoding='utf-8') as fh:
        for s in range(0, len(todo), shard):
            ids = todo[s:s + shard]
            probs, bs = score_shard(scorer, df, ids, batch_size)
            for i, p in zip(ids, probs):
                fh.write(json.dumps({'row': i, 'prob': float(p)} if bs == batch_size else
                                    {'row': i, 'prob': float(p), 'bs': bs}) + '\n')
            fh.flush()
            if time.time() - START > TIME_BUDGET_H * 3600:
                print(f'time budget of {TIME_BUDGET_H} h reached: stopping; run again with this output attached to finish')
                return
            rate = (s + len(ids)) / (time.time() - t0)
            print(f'  {s + len(ids)}/{len(todo)}  {rate:.1f} rows/s  about {(len(todo) - s - len(ids)) / rate / 60:.0f} min left',
                  flush=True)

def bacc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.nanmean([(p[y == 1] == 1).mean(), (p[y == 0] == 0).mean()]) * 100)

def summary(checker, split):
    if not os.path.exists(cache_path(checker, split)):
        print(f'{checker} {split}: nothing scored yet')
        return
    df = load_split(split)
    probs = {}
    for l in open(cache_path(checker, split), encoding='utf-8'):
        r = json.loads(l)
        probs[r['row']] = r['prob']
    missing = len(df) - len(probs)
    df = df[df.index.isin(probs)].copy()
    df['pred'] = [int(probs[i] > 0.5) for i in df.index]
    per = {ds: round(bacc(g.label, g.pred), 1) for ds, g in df.groupby('dataset')}
    avg = round(sum(per.values()) / len(per), 2)
    news3 = [per[d] for d in ('AggreFact-CNN', 'AggreFact-XSum', 'TofuEval-MediaS') if d in per]
    print(f'{checker} {split}: {len(df)} rows scored, {missing} missing')
    print(per)
    print('11-dataset average:', avg, '| News-3 mean:', round(sum(news3) / len(news3), 2) if news3 else None)
    if missing == 0 and split == 'test' and checker == 'minicheck-ft5':
        print('S1 gap vs the published 75.0:', round(avg - 75.0, 2), '(pass if within +/-2.0)')

print('ready')

## Part 1: S1, MiniCheck-Flan-T5-Large on the test split

In [ ]:
score_split('minicheck-ft5', 'test')
summary('minicheck-ft5', 'test')

## Part 2: MiniCheck-Flan-T5-Large on the dev split (for CAFG tuning)

In [ ]:
score_split('minicheck-ft5', 'dev')
summary('minicheck-ft5', 'dev')

### Part 2b (optional): MiniCheck-DeBERTa-v3-Large on test and dev (runs only if `RUN_DEBERTA = True` in 0.2)

In [ ]:
if RUN_DEBERTA:
    for split in ('test', 'dev'):
        score_split('minicheck-deberta', split)
        summary('minicheck-deberta', split)

## Part 3: Package the results

In [ ]:
import platform, subprocess, zipfile
import torch
env = {'tag': TAG, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'python': platform.python_version(),
       'pip_freeze': subprocess.run(['pip', 'freeze'], capture_output=True, text=True).stdout.splitlines(),
       'cache_files': {f: sum(1 for _ in open(f'{WORK}/cache/{f}', encoding='utf-8'))
                       for f in sorted(os.listdir(f'{WORK}/cache'))}}
gpu_tag = (env['gpu'] or 'cpu').split()[-1]
json.dump(env, open(f'{WORK}/outputs/environment_{TAG}_{gpu_tag}.json', 'w'), indent=1)
zip_path = f'{WORK}/mapnai_kaggle_results_{TAG}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in ('cache', 'outputs'):
        for f in os.listdir(f'{WORK}/{d}'):
            z.write(f'{WORK}/{d}/{f}', f'{d}/{f}')
print(json.dumps(env['cache_files'], indent=1))
print('Download this file from the notebook Output tab:', zip_path)